# Notebook 4: Electromagnetism — Velocity-Dependent Potentials & Lorentz Force

## Course: AP Physics C & Calculus BC Advanced Mechanics

### Objectives:
1. Understand why velocity-dependent magnetic forces require a **generalized potential** $V = q\Phi - q\mathbf{v}\cdot\mathbf{A}$.
2. Formulate the Electromagnetic Lagrangian $L = \frac{1}{2}m v^2 - q\Phi + q\mathbf{v}\cdot\mathbf{A}$.
3. Differentiate between **Canonical Momentum** $\mathbf{p} = m\mathbf{v} + q\mathbf{A}$ and **Kinematic Momentum** $\mathbf{\pi} = m\mathbf{v}$.
4. Use the **multivariable chain rule** to verify how scalar calculus naturally derives the Lorentz force $m\mathbf{a} = q(\mathbf{E} + \mathbf{v}\times\mathbf{B})$.
5. Simulate the helical cyclotron motion of a charged particle in a magnetic field.


In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

sp.init_printing()


---
## 1. Symbolic Multivariable Chain Rule Expansion for $A_x(x,y,z,t)$

The canonical momentum $x$-component is $p_x = \frac{\partial L}{\partial v_x} = m v_x + q A_x$.
Taking the total time derivative using Calculus BC multivariable chain rule:

$$\frac{d A_x}{dt} = \frac{\partial A_x}{\partial t} + v_x \frac{\partial A_x}{\partial x} + v_y \frac{\partial A_x}{\partial y} + v_z \frac{\partial A_x}{\partial z}$$

Equating with $\frac{\partial L}{\partial x}$ causes the $q v_x \frac{\partial A_x}{\partial x}$ terms to cancel perfectly, leaving the magnetic curl terms $(\nabla \times \mathbf{A})_x = B_x$.


In [ ]:
# Symbolic verification of Lorentz Force X-Component
t = sp.Symbol('t', real=True)
m, q = sp.symbols('m q', positive=True)
x, y, z = sp.symbols('x y z', real=True)
vx, vy, vz = sp.symbols('vx vy vz', real=True)

# Potentials
Phi = sp.Function('Phi')(x, y, z, t)
Ax = sp.Function('Ax')(x, y, z, t)
Ay = sp.Function('Ay')(x, y, z, t)
Az = sp.Function('Az')(x, y, z, t)

# Lagrangian
L = sp.Rational(1, 2) * m * (vx**2 + vy**2 + vz**2) - q * Phi + q * (vx*Ax + vy*Ay + vz*Az)

# Partial derivative wrt velocity vx
px = sp.diff(L, vx)
print('Canonical Momentum px = dL/dvx:')
sp.pprint(px)

# Partial derivative wrt position x
dL_dx = sp.diff(L, x)
print('\nGeneralized Force dL/dx:')
sp.pprint(dL_dx)


---
## 2. Numerical Simulation: Cyclotron Helical Trajectory in Magnetic Field

We simulate a particle with mass $m=1.0$, charge $q=1.0$ in a uniform magnetic field $\mathbf{B} = (0, 0, 2.0)$ and electric field $\mathbf{E} = (0, 0, 0.1)$.


In [ ]:
q_m = 1.0    # Charge to mass ratio q/m
B_z = 2.0    # Magnetic field along Z
E_z = 0.1    # Small accelerating Electric field along Z

def lorentz_ode(t, state):
    x, y, z, vx, vy, vz = state
    # F = q(E + v x B) -> a = (q/m)(E + v x B)
    ax = q_m * (vy * B_z)
    ay = q_m * (-vx * B_z)
    az = q_m * E_z
    return [vx, vy, vz, ax, ay, az]

state0 = [0.0, 0.0, 0.0, 2.0, 0.0, 0.5]
t_span = (0, 15.0)
t_eval = np.linspace(0, 15.0, 1000)

sol_em = solve_ivp(lorentz_ode, t_span, state0, t_eval=t_eval)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(sol_em.y[0], sol_em.y[1], sol_em.y[2], 'b-', linewidth=2, label='Charged Particle Helix')
ax.set_title('3D Helical Cyclotron Trajectory in EM Field', fontsize=14)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.legend(fontsize=11)
plt.savefig('/workspace/scratch/cyclotron_trajectory.png', dpi=150, bbox_inches='tight')
plt.close()
print('Cyclotron 3D plot generated successfully!')
